In [ ]:
import sys, os
sys.path = [os.path.dirname(os.getcwd())] + sys.path
print(sys.path)

In [ ]:
import rdkit
import rdkit.Chem as Chem
import rdkit.Chem.AllChem as AllChem
print(rdkit.rdBase.rdkitVersion)
from rdkit.Chem import AllChem

In [ ]:
import rdchiral
print(rdchiral.__path__)
from rdchiral.main import rdchiralRun, rdchiralRunText, rdchiralReaction, rdchiralReactants

In [ ]:
def sep_bar():
    print('')
    for i in range(3):
        print('='*80)
    print('')

In [ ]:
from rdkit.Chem.Draw import IPythonConsole 
from rdkit.Chem.Draw.MolDrawing import MolDrawing, DrawingOptions 

In [ ]:
reaction_smarts = '[C:1][OH:2]>>[C:1][O:2][C]'
reactant_smiles = 'CC(=O)OCCCO'
def show_outcomes(reaction_smarts, reactant_smiles):
    
    # normal version
    outcomes_rdkit_mol = AllChem.ReactionFromSmarts(reaction_smarts).RunReactants((Chem.MolFromSmiles(reactant_smiles),))
    outcomes_rdkit = set()
    for outcome in outcomes_rdkit_mol:
        outcomes_rdkit.add('.'.join(sorted([Chem.MolToSmiles(x) for x in outcome])))
    # rdchiral version
    outcomes_rdchiral = rdchiralRunText(reaction_smarts, reactant_smiles)
    
    print('Reaction SMARTS: {}'.format(reaction_smarts))
    display(Chem.MolFromSmiles(reactant_smiles))
    print('Input SMILES: {}'.format(reactant_smiles))
    
    if outcomes_rdkit:
        display(Chem.MolFromSmiles('.'.join(outcomes_rdkit)))
    print('{:1d} RDKit outcomes: {}'.format(len(outcomes_rdkit),'.'.join(outcomes_rdkit)))
    
    if outcomes_rdkit:
        display(Chem.MolFromSmiles('.'.join(outcomes_rdchiral)))
    print('{:1d} RDChiral outcomes: {}'.format(len(outcomes_rdchiral),'.'.join(outcomes_rdchiral)))
    
show_outcomes(reaction_smarts, reactant_smiles)

# Testing rdchiral and template extraction

This notebook looks at the performance of rdchiral in a number of cases where chirality is tricky to handle. There are many complications due to the many ways to write the same molecule.

In [ ]:
mols = [Chem.MolFromSmiles(x) for x in [
    'F[C@H](Br)Cl',
    'F[C@@H](Cl)Br',
    'Br[C@@H](F)Cl',
    'Br[C@H](Cl)F',
    'Cl[C@H](F)Br',
    'Cl[C@@H](Br)F'
]]

print('Num unique? {}'.format(len(set([Chem.MolToSmiles(mol, True) for mol in mols]))))
display(mols[0])

In [ ]:
mols = [Chem.MolFromSmiles(x) for x in [
    'F/C(Br)=C(/Cl)I',
    'F/C(Br)=C(Cl)\\I',
    'FC(/Br)=C(/Cl)I',
    'F\\C(Br)=C(\\Cl)I',
    'BrC(/F)=C(\\Cl)I',
    'BrC(/F)=C(Cl)/I',
]]

print('Num unique? {}'.format(len(set([Chem.MolToSmiles(mol, True) for mol in mols]))))
display(mols[0])

# Testing achiral transformations with achiral molecules

If our template is achiral and our molecule is too, we should get the same results as using the standard RDKit RunReactants

In [ ]:
# Preparing a carboxylic acid from hydrolysis of an ethyl ester
reaction_smarts = '[C:1](=[O:3])[OH:2]>>[C:1](=[O:3])[O:2]CC'
reactant_smiles = 'OC(=O)CCCCCC'
show_outcomes(reaction_smarts, reactant_smiles)

# Testing achiral transformations with chiral molecules (not in template)

When applying a transformation expecting an achiral reaction center to a molecule with specified chirality that does not overlap with the template, we should always preserve that chirality exactly. Even if the CIP priority of a branch from a tetrahedral center changes, the orientation stay the same.

In [ ]:
# Preparing a carboxylic acid from hydrolysis of an ethyl ester
reaction_smarts = '[C:1](=[O:3])[OH:2]>>[C:1](=[O:3])[O:2]CC'
reactant_smiles = 'OC(=O)CCCC[C@H](Cl)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'OC(=O)CCCC[C@@H](Cl)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'OC(=O)C[C@H](CC(=O)OC)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'OC(=O)C[C@H](CC(=O)OCCC)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'OC(=O)CC/C=C(F)\C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'OC(=O)CC/C=C(F)/C'
show_outcomes(reaction_smarts, reactant_smiles)

# Testing achiral transformations with chiral molecules (partially in template, but auxiliary)

When applying a transformation expecting an achiral reaction center to a molecule with specified chirality, we do _not_ want a match *unless* it would be impossible for the template to have specified chirality. For example, a tetrahedral center in a molecule matches part of a template, but that template does not contain all of that atom's neighbors. It would be impossible for the template to constrain the chirality, and thus we are okay matching. That tetrahedral center should be preserved.

In [ ]:
# Preparing a carboxylic acid from hydrolysis of an ethyl ester
reaction_smarts = '[C:4][C:1](=[O:3])[OH:2]>>[C:4][C:1](=[O:3])[O:2]CC'
reactant_smiles = 'OC(=O)[C@H](Cl)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'OC(=O)[C@@H](Cl)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'OC(=O)[C@@H](C(=O)OC)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'OC(=O)[C@H](C(=O)OC)C'
show_outcomes(reaction_smarts, reactant_smiles)

In [ ]:
# Alkylation reaction with unspecified chirality, template could not have specified
reaction_smarts = '[CH:1][O:2][C:3]>>[CH:1][OH:2].O[C:3]'
reactant_smiles = 'CCCC[C@@H](OCC)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'CC=C/C(OCC)C'
show_outcomes(reaction_smarts, reactant_smiles)

In [ ]:
# Alkylation reaction with unspecified chirality, template could not have specified due to symmetry
reaction_smarts = '[*:4][C:1]([*:5])[O:2][C:3]>>[*:4][C:1]([*:5])[OH:2].O[C:3]'
reactant_smiles = 'CCCC[C@@](CC)(OCC)C'
show_outcomes(reaction_smarts, reactant_smiles)

In [ ]:
reaction_smarts = '[OH:1][CH:2]=[C:3]>>CC[O:1][CH:2]=[C:3]'
reactant_smiles = 'O/C=C/CC'
show_outcomes(reaction_smarts, reactant_smiles)

# Testing achiral transformations with chiral molecules (fully in template)

When applying a transformation expecting an achiral reaction center to a molecule with specified chirality, we do _not_ want a match. Generating a retrosynthetic suggestion in this situation would falsely imply that the transformation could be performed stereoselectively.

In [ ]:
# Alkylation reaction with unspecified chirality, template could have specified
reaction_smarts = '[C:1][CH:2]([CH3:3])[O:4][C:5]>>[C:1][CH:2]([CH3:3])[OH:4].O[C:5]'
reactant_smiles = 'CCCC[C@@H](OCC)C'
show_outcomes(reaction_smarts, reactant_smiles)

In [ ]:
# Etherification reaction with unspecified chirality, template could have specified
reaction_smarts = '[CH:1]([CH3:2])=[CH:3]([CH2:4][O:5][C:6])>>[CH:1]([CH3:2])=[CH:3][CH2:4][OH:5].O[C:6]'
reactant_smiles = 'C(\C)=C\COCC'
show_outcomes(reaction_smarts, reactant_smiles)

In [ ]:
# Etherification reaction with unspecified chirality, template could have specified
reaction_smarts = '[C:1][CH:2]([CH3:3])[O:4][C:5]>>[C:1][CH:2]([CH3:3])[OH:4].O[C:5]'
reactant_smiles = 'CCCC[C@@H](OCC)C'
show_outcomes(reaction_smarts, reactant_smiles)

# Testing chiral transformations with achiral molecules

When the transformation expects a chiral reaction center but the input molecule is achiral, the template should not apply. In the retrosynthetic direction, this means that a stereospecific reaction should not be suggested as a way of synthesizing a racemic product.

In [ ]:
# SN2 with inversion of a tetrahedral center
reaction_smarts = '[C:1][C@H:2]([CH3:3])[I:4]>>[C:1][C@@H:2]([CH3:3])Br'
reactant_smiles = 'CCCCC(I)C'
show_outcomes(reaction_smarts, reactant_smiles)

In [ ]:
# Prepare a cis alkene from an alkyne
reaction_smarts = '[C:1]/[CH:2]=[CH:3]\[C:4]>>[C:1][C:2]#[C:3][C:4]'
reactant_smiles = 'CCCC=CCC'
show_outcomes(reaction_smarts, reactant_smiles)

# Testing chiral transformations with chiral molecules

Here is where the cases get particularly tricky. They've been broken into a number of subsections depending on the type of transformation.

### Reaction expects cis double bond

#### Molecule has explicit cis double bond

In [ ]:
# Prepare a cis alkene from an alkyne
reaction_smarts = '[C:1]/[CH:2]=[CH:3]\[C:4]>>[C:1][C:2]#[C:3][C:4]'
reactant_smiles = 'CCC/C=C\CC' # explicit cis
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'CCC/C=C/CC' # explicit trans
show_outcomes(reaction_smarts, reactant_smiles)

#### Molecule has an implicit double bond inside a ring

In [ ]:
# Prepare a cis alkene from an alkyne
reaction_smarts = '[C:1]/[CH:2]=[CH:3]\[C:4]>>[C:1][C:2]#[C:3][C:4]'
reactant_smiles = 'C1(CCC)CCCC=CCC1' # implicit cis
show_outcomes(reaction_smarts, reactant_smiles)

### Reaction expects trans double bond

In [ ]:
# Prepare a trans alkene from an alkyne
reaction_smarts = '[C:1]/[CH:2]=[CH:3]/[C:4]>>[C:1][C:2]#[C:3][C:4]'
reactant_smiles = 'C1(CCC)CCCC=CCC1' # implicit cis
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'CCC/C=C\CC' # explicit cis
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'CCC/C=C/CC' # explicit trans
show_outcomes(reaction_smarts, reactant_smiles)

### Reaction expects tetrahedral center

#### ...that will be inverted

In [ ]:
# SN2 with inversion of a tetrahedral center
reaction_smarts = '[C:1][C@H:2]([CH3:3])[I:4]>>[C:1][C@@H:2]([CH3:3])Br'
reactant_smiles = 'CCCC[C@@H](I)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reaction_smarts = '[C:1][C@H:2]([CH3:3])[I:4]>>[C:1][C@H:2](Br)[CH3:3]'
reactant_smiles = 'CCCC[C@@H](I)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reaction_smarts = '[C:1][C@@H:2]([CH3:3])[I:4]>>[C:1][C@@H:2](Br)[CH3:3]'
reactant_smiles = 'CCCC[C@@H](I)C'
show_outcomes(reaction_smarts, reactant_smiles)

#### ...that will be preserved
In retro direction, this reaction is stereoretentive

In [ ]:
# SN2 with retention of a tetrahedral center
reaction_smarts = '[C:1][C@H:2]([CH3:3])[I:4]>>[C:1][C@H:2]([CH3:3])Br'
reactant_smiles = 'CCCC[C@@H](I)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reaction_smarts = '[C:1][C@@H:2]([CH3:3])[I:4]>>[C:1][C@@H:2]([CH3:3])Br'
reactant_smiles = 'CCCC[C@@H](I)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reaction_smarts = '[C:1][C@H:2]([CH3:3])[I:4]>>[C:1][C@@H:2](Br)[CH3:3]'
reactant_smiles = 'CCCC[C@@H](I)C'
show_outcomes(reaction_smarts, reactant_smiles)

#### ...that will be destroyed
In retro direction, this reaction is stereospecific

In [ ]:
reaction_smarts = '[C:1][C@H:2]([CH3:3])[I:4]>>[C:1][CH:2](Br)[CH3:3]'
reactant_smiles = 'CCCC[C@@H](I)C'
show_outcomes(reaction_smarts, reactant_smiles)

#### ....that will be introduced
In retro direction, this reaction produces a racemic product from a chiral precursor

In [ ]:
reaction_smarts = '[C:1][CH:2]([CH3:3])[I:4]>>[C:1][C@H:2](Br)[CH3:3]'
reactant_smiles = 'CCCCC(I)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reaction_smarts = '[C:1][CH:2]([CH3:3])[I:4]>>[C:1][C@@H:2](Br)[CH3:3]'
reactant_smiles = 'CCCCC(I)C'
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reaction_smarts = '[C:1][CH:2]([CH3:3])[I:4]>>[C:1][C@H:2]([CH3:3])Br'
reactant_smiles = 'CCCCC(I)C'
show_outcomes(reaction_smarts, reactant_smiles)

In [ ]:
# Chirality specified in precursor, but symmetry makes them equivalent
reaction_smarts = '[C:1][CH:2]([C:3])[I:4]>>[C:1][C@H:2]([C:3])Br'
reactant_smiles = 'CCCCC(I)C'
show_outcomes(reaction_smarts, reactant_smiles)

### Molecule needs two stereocenters, diastereoselective

In [ ]:
# preparing a trans epoxide from a trans alkene
reaction_smarts = '[c:1]-[CH;@@;D3;+0:2]1-[O;H0;D2;+0]-[CH;@;D3;+0:3]-1-[c:4]>>[c:1]/[CH;D2;+0:2]=[CH;D2;+0:3]/[c:4]'
reactant_smiles = 'c1ccccc1[C@H]2[C@H](O2)c1ccccc1' # cis epoxide, should not match
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'c1ccccc1[C@H]2[C@@H](O2)c1ccccc1' # trans epoxide, should  match
show_outcomes(reaction_smarts, reactant_smiles)

In [ ]:
# preparing a cis epoxide from a cis alkene
reaction_smarts = '[c:1]-[CH;@@;D3;+0:2]1-[O;H0;D2;+0]-[CH;@@;D3;+0:3]-1-[c:4]>>[c:1]/[CH;D2;+0:2]=[CH;D2;+0:3]\[c:4]'
reactant_smiles = 'c1ccccc1[C@H]2[C@H](O2)c1ccccc1' # cis epoxide, should match
show_outcomes(reaction_smarts, reactant_smiles)
sep_bar()
reactant_smiles = 'c1ccccc1[C@H]2[C@@H](O2)c1ccccc1' # trans epoxide, should not match
show_outcomes(reaction_smarts, reactant_smiles)

In [ ]:
# Multiple prods?
# Intramolecular esterification (in forward direction)
reaction_smarts = '[C:1](=[O:3])[O:2][C:4]>>[C:1](=[O:3])[OH:2].O[C:4]'
reactant_smiles = 'C1C(=O)OCCC1'
show_outcomes(reaction_smarts, reactant_smiles)